In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import gc
import torch
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma

# Persistent storage directory
CHROMA_PATH = "/content/drive/MyDrive/VectorDB"

def get_jina_embeddings():
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading Jina Embeddings on: {device}")

    embeddings = HuggingFaceEmbeddings(
        model_name="jinaai/jina-embeddings-v2-base-en",
        model_kwargs={'device': device, 'trust_remote_code': True},
        encode_kwargs={'normalize_embeddings': True}
    )
    return embeddings

def process_and_store_pdfs(pdf_directory: str, vector_db_path: str):
    embeddings = get_jina_embeddings()

    # Massive Chunking Strategy
    # Leveraging the 8192 token window.
    # 20,000 chars is roughly 5,000 - 6,000 tokens, keeping us safely under the limit
    # while embedding massive, continuous blocks of regulatory text.
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=20000,
        chunk_overlap=1000,
        separators=["\n\n\n", "\n\n", "\n", ".", " ", ""],
        length_function=len
    )

    vector_db = Chroma(
        embedding_function=embeddings,
        persist_directory=vector_db_path
    )

    pdf_files = [f for f in os.listdir(pdf_directory) if f.endswith('.pdf')]
    batch_size = 3

    for i in range(0, len(pdf_files), batch_size):
        batch = pdf_files[i:i + batch_size]
        all_splits = []

        for file in batch:
            print(f"Parsing: {file}")
            loader = PyPDFLoader(os.path.join(pdf_directory, file))
            docs = loader.load()

            # Attach Metadata for exact citations
            for i, doc in enumerate(docs):
                doc.metadata["source"] = file
                doc.metadata["page"] = i + 1

            splits = text_splitter.split_documents(docs)
            all_splits.extend(splits)

        print(f"Embedding {len(all_splits)} massive chunks into ChromaDB...")
        # This is where the T4 GPU does the heavy lifting
        vector_db.add_documents(all_splits)

        del all_splits
        gc.collect()
        torch.cuda.empty_cache()
        print("GPU VRAM cleared for next batch.\n")

    print(f"Complete! Database permanently saved to {vector_db_path}")
    return vector_db

# Execution
pdf_folder = "/content/pdfs"
db = process_and_store_pdfs(pdf_folder, CHROMA_PATH)

ModuleNotFoundError: No module named 'langchain_community'

In [ ]:
!pip uninstall -y torch torchvision transformers sentence-transformers -q

!pip install -q \
torch==2.3.1 \
torchvision==0.18.1 \
transformers==4.44.2 \
sentence-transformers==3.0.1 \
langchain \
langchain-community \
langchain-text-splitters \
langchain-huggingface \
chromadb \
pypdf \
accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 4.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 779.1/779.1 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.0/7.0 MB 85.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 70.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 227.1/227.1 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 80.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 60.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 731.7/731.7 MB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 MB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56

In [ ]:
import os

print(os.listdir("/content/drive/MyDrive/VectorDB"))

FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/VectorDB'

In [ ]:
from transformers.onnx import OnnxConfig
print("Transformers OK")

from sentence_transformers import SentenceTransformer
print("SentenceTransformers OK")

import torch
print(torch.__version__)
print(torchvision.__version__)

Transformers OK
SentenceTransformers OK
2.3.1+cu121


NameError: name 'torchvision' is not defined